# 경북대학교 내 가로수 여부 확인

## 1) import 및 데이터 불러오기

In [17]:
import pandas as pd
import numpy as np

In [18]:
file = r'C:\Users\1\Desktop\proj. SENA\공모전\대구 빅데이터 분석대회\추가 데이터\대구광역시_가로수정보(31종-포털등록용)_20240418.csv'

df = pd.read_csv(file, encoding = 'cp949')
df.head()

,관리번호,수종,수목구분,수목유형,식재일,관리기관전화번호,관리기관명,데이터기준일자,도로명,도로종류,...,수령,보호덮개,보호틀,보호틀파손여부,뿌리융기상태,통기,지장물1,지장물2,수목사진,보호덮개사진
0,05-구리로-가-0266,중국단풍,가로수,교목,1982-09-25,053-665-2858,대구광역시 북구청,2022-12-09,구리로,광역시도,...,45.0,NaN,사각형,N,낮음,N,고압선,가로등,05-구리로-가-0266-001.jpg,05-구리로-가-0266-002.jpg
1,05-구리로-가-0265,중국단풍,가로수,교목,1978-09-25,053-665-2858,대구광역시 북구청,2022-12-09,구리로,광역시도,...,49.0,주철,사각형,N,NaN,N,고압선,전신주,05-구리로-가-0265-001.jpg,05-구리로-가-0265-002.jpg
2,05-구리로-가-0262,중국단풍,가로수,교목,1968-09-25,053-665-2858,대구광역시 북구청,2022-12-09,구리로,광역시도,...,59.0,NaN,사각형,N,NaN,N,고압선,전깃줄,05-구리로-가-0262-001.jpg,05-구리로-가-0262-002.jpg
3,05-학남로3길-가-0011,은행나무,가로수,교목,1998-04-15,053-665-2858,대구광역시 북구청,2022-12-09,학남로3길,광역시도,...,29.0,NaN,NaN,N,NaN,N,전신주,고압선,05-학남로3길-가-0011-001.jpg,05-학남로3길-가-0011-002.jpg
4,05-구리로34길-가-0003,단풍나무,가로수,교목,1952-09-30,053-665-2858,대구광역시 북구청,2022-12-09,구리로34길,광역시도,...,75.0,NaN,사각형,Y,NaN,N,표지판,전깃줄,05-구리로34길-가-0003-001.jpg,05-구리로34길-가-0003-002.jpg


In [19]:
df.columns

Index(['관리번호', '수종', '수목구분', '수목유형', '식재일', '관리기관전화번호', '관리기관명', '데이터기준일자',
       '도로명', '도로종류', '도로시작점', '도로종료점', '수목위치', '수목위도', '수목경도', '도로명주소',
       '지번주소', '행정동', '시군구', '흉고직경', '근원직경', '수령', '보호덮개', '보호틀', '보호틀파손여부',
       '뿌리융기상태', '통기', '지장물1', '지장물2', '수목사진', '보호덮개사진'],
      dtype='object')

## 2) Bounding box (lat, lon) 설정 및 필터링

In [20]:
LAT_MIN, LAT_MAX = 35.8792, 35.8933
LON_MIN, LON_MAX = 128.6013, 128.6206

In [21]:
mask = (
    df["수목위도"].between(LAT_MIN, LAT_MAX) &
    df["수목경도"].between(LON_MIN, LON_MAX)
)

In [22]:
df = df[mask].copy()

In [23]:
len(df)

1200

## 3) 높이 너비 추정
* 단순 경험식: 높이 = DBH(cm) * 0.5, 너비 = DBH(cm) * 0.15

In [24]:
dbh_cm = df["흉고직경"].fillna(20)  # cm 단위, 결측값은 20cm로 가정
df["추정높이_m"] = np.clip(dbh_cm * 0.5, 2.5, 35)     # 최소~최대 제한
df["추정너비_m"] = np.clip(dbh_cm * 0.15, 1.0, 20)

## 4) 필요 컬럼만 필터링 및 저장

In [26]:
result = df[["관리번호", "수종", "수목위도", "수목경도", "추정높이_m", "추정너비_m"]]

In [27]:
result.head()

,관리번호,수종,수목위도,수목경도,추정높이_m,추정너비_m
315,05-대현로-가-0188,왕벚나무,35.887015,128.602386,2.5,1.0
316,05-연암로-가-0323,왕벚나무,35.888324,128.601696,2.5,1.0
318,05-대현로-가-0184,양버즘나무,35.886475,128.602523,2.5,1.0
319,05-대현로-가-0106,양버즘나무,35.882006,128.611598,2.5,1.0
320,05-대현로-가-0032,소나무,35.884617,128.604996,2.5,1.0


저장

In [28]:
result.to_csv("가로수_shadow_ready.csv", index=False, encoding="utf-8-sig")

## 5) 시각화

In [29]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster

지도 초기 위치(평균값)

In [31]:
lat0 = df["수목위도"].mean()
lon0 = df["수목경도"].mean()
m = folium.Map(location=[lat0, lon0], zoom_start=16, tiles="cartodbpositron")

In [32]:
LAT_MIN, LAT_MAX = 35.8792, 35.8933
LON_MIN, LON_MAX = 128.6013, 128.6206

In [34]:
bbox = folium.Polygon(
    locations=[(LAT_MIN, LON_MIN), (LAT_MIN, LON_MAX),
               (LAT_MAX, LON_MAX), (LAT_MAX, LON_MIN)],
    color="#2b8cbe", weight=2, fill=True, fill_opacity=0.05
)

folium.FeatureGroup(name="범위(BBox)").add_child(bbox).add_to(m)

In [35]:
fg_canopy = folium.FeatureGroup(name="가로수 수관(지름=추정너비_m)")
mc = MarkerCluster(name="가로수 포인트").add_to(m)

In [36]:
for _, row in df.iterrows():
    lat, lon = row["수목위도"], row["수목경도"]
    h, d = row["추정높이_m"], row["추정너비_m"]
    species = str(row.get("수종", "미상"))
    gid = str(row.get("관리번호", ""))

    # 수관 원 (반지름 = 지름/2, 단위: 미터)
    if pd.notna(d):
        folium.Circle(
            location=(lat, lon),
            radius=float(d) / 2.0,     # m
            color="#31a354",
            weight=1,
            fill=True,
            fill_opacity=0.25
        ).add_to(fg_canopy)

    # 포인트 마커 (팝업)
    popup_html = f"""
    <b>관리번호</b>: {gid}<br>
    <b>수종</b>: {species}<br>
    <b>위도/경도</b>: {lat:.6f}, {lon:.6f}<br>
    <b>추정 높이</b>: {h:.1f} m<br>
    <b>추정 수관 지름</b>: {d:.1f} m
    """
    folium.Marker(
        location=(lat, lon),
        popup=folium.Popup(popup_html, max_width=300),
        icon=folium.Icon(color="green", icon="tree-conifer", prefix="glyphicon")
    ).add_to(mc)

In [37]:
fg_canopy.add_to(m)
folium.LayerControl().add_to(m)

In [40]:
m.save("가로수_map.html")